# Why 6D rotations beat Euler angles

**Track A · Human Modeling** · maps to lesson A6 (rotation continuity).

Euler angles (and quaternions) are **discontinuous** as a network output, which hurts learning. The **6D** representation (Zhou et al. 2019) is continuous. We train identical nets to regress random rotations with each parametrisation and compare the geodesic error.

> CPU is fine.

In [ ]:
import os, torch, torch.nn as nn, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 3000))
def rand_rot(n):
    q = torch.randn(n, 4, device=device); q = q / q.norm(dim=1, keepdim=True)
    w, x, y, z = q.unbind(1)
    return torch.stack([1-2*(y*y+z*z), 2*(x*y-z*w), 2*(x*z+y*w),
                        2*(x*y+z*w), 1-2*(x*x+z*z), 2*(y*z-x*w),
                        2*(x*z-y*w), 2*(y*z+x*w), 1-2*(x*x+y*y)], 1).view(n, 3, 3)
pts = torch.randn(8, 3, device=device)
def geo(A, B):
    tr = (A.transpose(1, 2) @ B).diagonal(dim1=1, dim2=2).sum(1)
    return torch.acos(((tr - 1) / 2).clamp(-1 + 1e-6, 1 - 1e-6))

## 1 · The two output heads

In [ ]:
def sixd_to_R(d):
    a1, a2 = d[:, :3], d[:, 3:]
    b1 = a1 / a1.norm(dim=1, keepdim=True)
    a2 = a2 - (b1 * a2).sum(1, keepdim=True) * b1; b2 = a2 / a2.norm(dim=1, keepdim=True)
    b3 = torch.cross(b1, b2, dim=1)
    return torch.stack([b1, b2, b3], 2)
def euler_to_R(e):
    cx, cy, cz = torch.cos(e).unbind(1); sx, sy, sz = torch.sin(e).unbind(1)
    return torch.stack([cy*cz, cz*sx*sy-cx*sz, sx*sz+cx*cz*sy,
                        cy*sz, cx*cz+sx*sy*sz, cx*sy*sz-cz*sx,
                        -sy, cy*sx, cx*cy], 1).view(-1, 3, 3)
def make_net(out):
    return nn.Sequential(nn.Linear(24, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, out)).to(device)

## 2 · Train both (same data, same budget)

In [ ]:
def train(out, to_R):
    net = make_net(out); opt = torch.optim.Adam(net.parameters(), 1e-3); h = []
    for step in range(STEPS + 1):
        R = rand_rot(256); x = (R @ pts.T).transpose(1, 2).reshape(256, -1)
        loss = geo(to_R(net(x)), R).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if step % max(1, STEPS // 10) == 0: h.append((step, loss.item()))
    return net, h
torch.manual_seed(0); _, h6 = train(6, sixd_to_R)
torch.manual_seed(0); m_e, hE = train(3, euler_to_R)
torch.manual_seed(0); m6, _ = train(6, sixd_to_R)
print(f"final mean geodesic error (rad) — 6D {h6[-1][1]:.3f}  vs  Euler {hE[-1][1]:.3f}")

## 3 · Compare — 6D converges lower

In [ ]:
import math
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(*zip(*h6), label="6D (continuous)"); ax.plot(*zip(*hE), label="Euler (discontinuous)")
ax.set_xlabel("step"); ax.set_ylabel("geodesic error (rad)"); ax.legend(); ax.grid(alpha=.3); ax.set_title("rotation representation matters")
plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/A_rotation_6d/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/A_rotation_6d"; os.makedirs(run, exist_ok=True)
torch.save(m6.state_dict(), f"{run}/rot6d.pt")
json.dump({"geo_6d": h6, "geo_euler": hE}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

### Where to go next
- Use 6D outputs anywhere a network predicts rotation: body pose (SMPL θ), camera pose, hand pose.
- Read Zhou et al., *On the Continuity of Rotation Representations* (CVPR 2019).